## Simple Calculator Evaluation
Here we can use the Python SDK to develop the simple calculator agent, then save the agent to a config.yaml and run it from there.

In [1]:
import os
import sys

# Import the NeMo-Agent-Toolkit module
module_path = os.path.abspath('../../../src/')
if module_path not in sys.path:
    sys.path.insert(0, module_path)

In [ ]:
from nat.agent.sdk import NatReActAgent
from nat.llm.sdk import NimLLM
from nat.tool.sdk import CurrentTimeTool
from nat.utils.sdk.nat_workflow import NatWorkflow
from nat_simple_calculator.sdk import CalculatorToolGroup

llm = NimLLM(
    model_name="nvdev/meta/llama-3.1-70b-instruct",
    temperature=0.0,
    max_tokens=1024,
    name="nim_llm",
)

current_time_tool = CurrentTimeTool(
    name="current_datetime",
)

calculator_tool_group = CalculatorToolGroup(
    name="calculator",
)

agent = NatReActAgent(
    tools=[current_time_tool, calculator_tool_group],
    llm=llm,
    verbose=True,
    parse_agent_response_max_retries=3,
)

nat_workflow = NatWorkflow(
    entrypoint=agent,
)

/Users/spastoriza/Documents/Programming/public/nat-fork/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import os
from pathlib import Path

path_to_yaml = Path(os.getcwd(), "config", "config.yaml").resolve()

# Create the config directory if it doesn't exist
if not path_to_yaml.parent.exists():
    os.makedirs(path_to_yaml.parent)

# Save the workflow to a config file
nat_workflow.save_to_config_file(path_to_yaml)

# Print out the config file content
with open(path_to_yaml) as f:
    print(f.read())

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


functions:
  current_datetime:
    _type: current_datetime

function_groups:
  calculator:
    _type: calculator

llms:
  nim_llm:
    _type: nim
    model: nvdev/meta/llama-3.1-70b-instruct
    max_tokens: 1024
    temperature: 0.0

workflow:
  _type: react_agent
  llm_name: nim_llm
  verbose: true
  tool_names:
  - current_datetime
  - calculator
  parse_agent_response_max_retries: 3



In [ ]:
await nat_workflow.prompt('What is 4 * 50 plus the current hour?')

'214.0'

In [5]:
judge_prompt = """
You are an intelligent evaluator that scores the generated answer based on the description of the expected answer.
The score is a measure of how well the generated answer matches the description of the expected answer based on the question.
Take into account the question, the relevance of the answer to the question and the quality compared to the description of the expected answer.

Rules:
- The score must be a float of any value between 0.0 and 1.0 on a sliding scale.
- The reasoning string must be concise and to the point. It should be 1 sentence and 2 only if extra description is needed. It must explain why the score was given and what is different between the generated answer and the expected answer.
- The tags <image> and <chart> are real images and charts.
"""  # noqa: E501


In [ ]:
from pathlib import Path

from nat.eval.sdk import TunableRagEvaluator
from nat.llm.sdk import NimLLM
from nat.utils.sdk.nat_evaluation import EvalDatasetJsonConfig
from nat.utils.sdk.nat_evaluation import NatEvaluation

path_to_dataset = Path(os.path.curdir,
                       "../../../../",
                       "examples/getting_started/simple_calculator/src/nat_simple_calculator/data/simple_calculator.json").resolve()

evaluator_llm = NimLLM(
    model_name="nvdev/mistralai/mixtral-8x22b-instruct-v0.1",
    temperature=0.0,
    max_tokens=1024,
    name="eval_llm",
)

tunable_rag_evaluator = TunableRagEvaluator(
    llm=evaluator_llm,
    default_scoring=True,
    default_score_weights={
        "coverage": 0.5,
        "consistency": 0.3,
        "relevance": 0.2
    },
    judge_llm_prompt=judge_prompt,
    name="tuneable_eval"
)

evaluation = NatEvaluation(
    max_concurrency=1,
    output_dir=Path(".tmp/nat/examples/getting_started/simple_calculator"),
    dataset=EvalDatasetJsonConfig(file_path=path_to_dataset),
    evaluators=[tunable_rag_evaluator],
)

nat_workflow.add_evaluator(evaluation)

In [ ]:
path_to_yaml = Path(os.getcwd(), "config", "eval_config.yaml").resolve()

# Create the config directory if it doesn't exist
if not path_to_yaml.parent.exists():
    os.makedirs(path_to_yaml.parent)

# Save the workflow to a config file
nat_workflow.save_to_config_file(path_to_yaml)

# Print out the config file content
with open(path_to_yaml) as f:
    print(f.read())

Serialization failed: 1 validation error for EvalGeneralConfig
dataset
  Unable to extract tag using discriminator discriminator() [type=union_tag_not_found, input_value={'file_path': '/Users/spa...simple_calculator.json'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/union_tag_not_found


Traceback (most recent call last):
  File "/Users/spastoriza/Documents/Programming/public/nat-fork/src/nat/utils/sdk/nat_agent.py", line 143, in save_to_config_file
    serialized = self._config.model_dump(exclude_unset=True, by_alias=True, round_trip=True)
                 ^^^^^^^^^^^^
  File "/Users/spastoriza/miniconda3/lib/python3.13/functools.py", line 1026, in __get__
    val = self.func(instance)
  File "/Users/spastoriza/Documents/Programming/public/nat-fork/src/nat/utils/sdk/nat_agent.py", line 96, in _config
    return self.__build_config_object()
           ~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "/Users/spastoriza/Documents/Programming/public/nat-fork/src/nat/utils/sdk/nat_agent.py", line 356, in __build_config_object
    evaluator = self.build_evaluator()
  File "/Users/spastoriza/Documents/Programming/public/nat-fork/src/nat/utils/sdk/nat_agent.py", line 548, in build_evaluator
    eval_config["general"] = EvalGeneralConfig(**self.evaluator.general_evaluator.model_dump(
     

FileNotFoundError: [Errno 2] No such file or directory: '/Users/spastoriza/Documents/Programming/public/nat-fork/examples/evaluation_and_profiling/simple_calculator_eval/notebook/config/eval_config.yaml'

In [ ]:
await nat_workflow.evaluate()

Evaluating RAG: 100%|██████████| 12/12 [00:14<00:00,  1.21s/it]
